# Static DU — Raw Output Extraction (Java)

This notebook performs **end-to-end execution of the Static DU (Definition-Use Analysis) tool** against a Java repository and captures the **complete raw tool output** without modifying or filtering the results.

**Default repository:** [visvantha-testable/java-tool-testing-static-du](https://github.com/visvantha-testable/java-tool-testing-static-du)

> **Important:** Metrics are parsed only from explicit Static DU console and JSON outputs. Copied JSON files are preserved exactly as generated by the tool.


## Section 1 — Install Dependencies


In [ ]:
import platform
import shutil
import subprocess
import sys

IS_COLAB = 'google.colab' in sys.modules
IS_LINUX = platform.system() == 'Linux'

if IS_COLAB or IS_LINUX:
    !apt-get update -qq
    !apt-get install -y openjdk-17-jdk maven

!pip install -q pandas gitpython notebook jupyter

print('Java:')
subprocess.run(['java', '-version'], check=False)
if shutil.which('mvn'):
    print('\nMaven:')
    subprocess.run(['mvn', '-version'], check=False)
gradlew = shutil.which('gradlew')
if gradlew:
    print('\nGradle wrapper found:', gradlew)


## Section 2 — Configuration


In [ ]:
USE_GIT_REPO = True

REPO_URL = 'https://github.com/visvantha-testable/java-tool-testing-static-du.git'

LOCAL_REPO = './workspace/java-tool-testing-static-du'

WORKSPACE_DIR = './workspace'
OUTPUT_DIR = './outputs'
IF_CLONE_EXISTS = 'reuse'
CLONE_DEPTH = 1
RAW_OUTPUT_PREVIEW_LINES = 150
SKIP_VERIFY = True

# Local mode example:
# USE_GIT_REPO = False
# LOCAL_REPO = './workspace/java-tool-testing-static-du'

# Colab example:
# USE_GIT_REPO = False
# LOCAL_REPO = '/content/java-tool-testing-static-du'


## Section 3 — Imports and Utility Functions


In [ ]:
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

JACOCO_UTILS_ROOT = Path('..') / 'JaCoCo Coverage' / 'tool'
sys.path.insert(0, str(JACOCO_UTILS_ROOT.resolve()))
sys.path.insert(0, str(Path('tool').resolve()))

from __future__ import annotations

import csv
import os
import platform
import re
import shutil
import subprocess
import sys
import time
import xml.etree.ElementTree as ET
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from git import Repo
from git.exc import GitCommandError, InvalidGitRepositoryError

EXCLUDED_DIR_NAMES = {
    ".git", "target", "build", "bin", ".idea", ".gradle", ".mvn", "out", "node_modules",
}
COUNTER_TYPES = ["INSTRUCTION", "BRANCH", "LINE", "METHOD", "CLASS", "COMPLEXITY"]
PY = sys.executable
PACKAGE_RE = re.compile(r"^\s*package\s+([\w.]+)\s*;", re.MULTILINE)


@dataclass
class CommandResult:
    command: list[str]
    stdout: str
    stderr: str
    exit_code: int
    execution_time_seconds: float


@dataclass
class BuildStatus:
    build_tool: str
    build_command: list[str]
    jacoco_command: list[str]
    build_result: CommandResult | None = None
    jacoco_result: CommandResult | None = None
    build_success: bool = False
    test_success: bool = False
    report_generated: bool = False
    report_dir: Path | None = None
    jacoco_exec: Path | None = None
    jacoco_xml: Path | None = None
    jacoco_csv: Path | None = None
    index_html: Path | None = None


class NotebookLogger:
    def __init__(self, error_log_path: Path) -> None:
        self.error_log_path = error_log_path
        self.error_log_path.parent.mkdir(parents=True, exist_ok=True)
        self._errors: list[dict[str, str]] = []
        self.write_errors()

    def info(self, message: str) -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        print(f"[{timestamp}] INFO: {message}")

    def error(self, message: str, step: str = "notebook") -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        line = f"[{timestamp}] ERROR: {message}\n"
        print(line, end="")
        self._errors.append({"timestamp": timestamp, "step": step, "error": message})
        self.write_errors()

    def write_errors(self) -> None:
        with self.error_log_path.open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=["timestamp", "step", "error"])
            writer.writeheader()
            writer.writerows(self._errors)


def ensure_output_dir(output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)


def project_runtimes_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    candidates: list[Path] = []
    for parent in [current, *current.parents]:
        candidate = parent / "runtimes"
        if candidate.is_dir():
            candidates.append(candidate)
    for candidate in candidates:
        if (candidate / "jdk-21").exists() or any(candidate.glob("apache-maven-*")):
            return candidate
    return candidates[0] if candidates else current / "runtimes"


def configure_java_runtime(logger: NotebookLogger) -> dict[str, str]:
    runtimes = project_runtimes_root()
    env = os.environ.copy()
    jdk_candidates = [
        runtimes / "jdk-21",
        Path(env.get("JAVA_HOME", "")),
        Path(r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"),
    ]
    for candidate in jdk_candidates:
        java_bin = candidate / "bin"
        java_exe = java_bin / ("java.exe" if platform.system() == "Windows" else "java")
        if java_exe.exists():
            env["JAVA_HOME"] = str(candidate)
            env["PATH"] = str(java_bin) + os.pathsep + env.get("PATH", "")
            logger.info(f"Using JAVA_HOME={candidate}")
            break
    else:
        logger.info("Using system Java from PATH.")
    return env


def java_version_text(env: dict[str, str]) -> str:
    completed = subprocess.run(
        ["java", "-version"],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=False,
        env=env,
    )
    return combine_streams(completed.stdout, completed.stderr).strip()


def resolve_maven_command(repo_path: Path, logger: NotebookLogger) -> list[str]:
    if platform.system() == "Windows":
        wrapper = repo_path / "mvnw.cmd"
        if wrapper.exists():
            return [str(wrapper)]
    else:
        wrapper = repo_path / "mvnw"
        if wrapper.exists():
            return [str(wrapper)]
    runtimes = project_runtimes_root(repo_path)
    bundled = runtimes / "apache-maven-3.9.6" / "bin" / ("mvn.cmd" if platform.system() == "Windows" else "mvn")
    if bundled.exists():
        logger.info(f"Using bundled Maven: {bundled}")
        return [str(bundled)]
    return ["mvn"]


def resolve_gradle_command(repo_path: Path, logger: NotebookLogger) -> list[str]:
    if platform.system() == "Windows":
        wrapper = repo_path / "gradlew.bat"
    else:
        wrapper = repo_path / "gradlew"
    if wrapper.exists():
        if platform.system() != "Windows":
            wrapper.chmod(wrapper.stat().st_mode | 0o111)
        logger.info(f"Using Gradle wrapper: {wrapper}")
        return [str(wrapper)]
    return ["gradle"]


def derive_clone_path(repo_url: str, workspace_dir: Path) -> Path:
    repo_name = repo_url.rstrip("/").removesuffix(".git").split("/")[-1]
    if not repo_name:
        raise ValueError(f"Unable to derive repository name from URL: {repo_url}")
    return workspace_dir / repo_name


def validate_repo_url(repo_url: str) -> None:
    if not repo_url or not isinstance(repo_url, str):
        raise ValueError("REPO_URL must be a non-empty string.")
    if not (repo_url.startswith("https://") or repo_url.startswith("git@") or repo_url.startswith("http://")):
        raise ValueError(f"Invalid repository URL format: {repo_url}")


def clone_or_reuse_repository(
    repo_url: str,
    workspace_dir: Path,
    if_clone_exists: str,
    logger: NotebookLogger,
    clone_depth: int | None = None,
) -> Path:
    validate_repo_url(repo_url)
    workspace_dir.mkdir(parents=True, exist_ok=True)
    clone_path = derive_clone_path(repo_url, workspace_dir)
    if clone_path.exists():
        if if_clone_exists == "reclone":
            logger.info(f"Removing existing clone at {clone_path}")
            shutil.rmtree(clone_path)
        elif if_clone_exists == "reuse":
            logger.info(f"Reusing existing clone at {clone_path}")
            return clone_path.resolve()
        else:
            raise ValueError('IF_CLONE_EXISTS must be either "reuse" or "reclone"')
    logger.info(f"Cloning {repo_url} into {clone_path}")
    clone_kwargs: dict[str, Any] = {"depth": clone_depth} if clone_depth else {}
    try:
        Repo.clone_from(repo_url, clone_path, **clone_kwargs)
    except GitCommandError as exc:
        logger.error(f"Git clone failed: {exc}", step="clone")
        raise
    return clone_path.resolve()


def validate_local_repo_path(local_repo_path: Path, logger: NotebookLogger) -> Path:
    if not local_repo_path.exists():
        msg = f"Local repository path does not exist: {local_repo_path}"
        logger.error(msg, step="repository")
        raise FileNotFoundError(msg)
    if not local_repo_path.is_dir():
        msg = f"Local repository path is not a directory: {local_repo_path}"
        logger.error(msg, step="repository")
        raise NotADirectoryError(msg)
    build_files = [
        local_repo_path / "pom.xml",
        local_repo_path / "build.gradle",
        local_repo_path / "build.gradle.kts",
    ]
    if not any(path.exists() for path in build_files):
        msg = "No pom.xml, build.gradle, or build.gradle.kts found in repository."
        logger.error(msg, step="repository")
        raise FileNotFoundError(msg)
    return local_repo_path.resolve()


def resolve_repository_path(
    use_git_repo: bool,
    repo_url: str,
    local_repo: str | Path,
    workspace_dir: Path,
    if_clone_exists: str,
    logger: NotebookLogger,
    clone_depth: int | None = None,
) -> Path:
    if use_git_repo:
        return clone_or_reuse_repository(repo_url, workspace_dir, if_clone_exists, logger, clone_depth)
    return validate_local_repo_path(Path(local_repo), logger)


def detect_build_tool(repo_path: Path) -> str:
    if (repo_path / "pom.xml").exists():
        return "Maven"
    if (repo_path / "build.gradle.kts").exists():
        return "Gradle Kotlin DSL"
    if (repo_path / "build.gradle").exists():
        return "Gradle"
    raise FileNotFoundError("Unable to detect Maven or Gradle build files.")


def discover_java_files(repo_path: Path) -> list[Path]:
    files: list[Path] = []
    for path in repo_path.rglob("*.java"):
        if any(part in EXCLUDED_DIR_NAMES for part in path.parts):
            continue
        files.append(path.resolve())
    return sorted(files)


def extract_package_name(java_path: Path) -> str:
    try:
        text = java_path.read_text(encoding="utf-8")
    except (OSError, UnicodeDecodeError):
        return ""
    match = PACKAGE_RE.search(text)
    return match.group(1) if match else ""


def save_java_inventory(java_files: list[Path], output_csv: Path) -> None:
    rows = [
        {
            "file_path": str(path),
            "file_name": path.name,
            "package": extract_package_name(path),
            "directory": str(path.parent),
        }
        for path in java_files
    ]
    pd.DataFrame(rows).to_csv(output_csv, index=False)


def count_all_files(repo_path: Path) -> int:
    total = 0
    for path in repo_path.rglob("*"):
        if path.is_file() and not any(part in EXCLUDED_DIR_NAMES for part in path.parts):
            total += 1
    return total


def compute_repository_stats(
    repo_path: Path,
    java_files: list[Path],
    build_tool: str,
    java_version: str,
) -> dict[str, Any]:
    total_size = sum(path.stat().st_size for path in java_files)
    return {
        "repository_name": repo_path.name,
        "repository_location": str(repo_path),
        "build_tool": build_tool,
        "java_version": java_version.replace("\n", " | "),
        "total_java_files": len(java_files),
        "total_files": count_all_files(repo_path),
        "repository_size_bytes": total_size,
    }


def combine_streams(stdout: str, stderr: str) -> str:
    raw = stdout
    if stderr:
        if raw and not raw.endswith("\n"):
            raw += "\n"
        raw += stderr
    return raw


def run_shell_command(
    command: list[str],
    cwd: Path,
    env: dict[str, str],
    logger: NotebookLogger,
    step: str,
) -> CommandResult:
    logger.info(f"Executing ({step}): {' '.join(command)}")
    started = time.perf_counter()
    completed = subprocess.run(
        command,
        cwd=str(cwd),
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=False,
        env=env,
    )
    elapsed = round(time.perf_counter() - started, 5)
    result = CommandResult(
        command=command,
        stdout=completed.stdout,
        stderr=completed.stderr,
        exit_code=completed.returncode,
        execution_time_seconds=elapsed,
    )
    if completed.returncode != 0:
        logger.error(
            combine_streams(completed.stdout, completed.stderr) or f"Command failed with exit code {completed.returncode}",
            step=step,
        )
    return result


def find_jacoco_report_dirs(repo_path: Path) -> list[Path]:
    report_dirs: list[Path] = []
    for xml_path in repo_path.rglob("jacoco.xml"):
        if ".git" in xml_path.parts:
            continue
        parent = str(xml_path.parent).replace("\\", "/")
        if "/site/jacoco" in parent or "/reports/jacoco" in parent or parent.endswith("/jacoco"):
            report_dirs.append(xml_path.parent.resolve())
    return sorted(set(report_dirs))


def find_jacoco_exec_files(repo_path: Path) -> list[Path]:
    exec_files: list[Path] = []
    for exec_path in repo_path.rglob("jacoco.exec"):
        if ".git" in exec_path.parts:
            continue
        exec_files.append(exec_path.resolve())
    return sorted(exec_files)


def choose_primary_report_dir(report_dirs: list[Path]) -> Path | None:
    if not report_dirs:
        return None
    best_dir = report_dirs[0]
    best_score = -1
    for report_dir in report_dirs:
        xml_path = report_dir / "jacoco.xml"
        if not xml_path.exists():
            continue
        counters = parse_counter_map(xml_path)
        score = sum(counters.get(counter, {}).get("covered", 0) + counters.get(counter, {}).get("missed", 0) for counter in COUNTER_TYPES)
        if score > best_score:
            best_score = score
            best_dir = report_dir
    return best_dir


def choose_primary_exec(exec_files: list[Path], report_dir: Path | None) -> Path | None:
    if report_dir is not None:
        maven_exec = report_dir.parent.parent / "jacoco.exec"
        if maven_exec.exists():
            return maven_exec.resolve()
        gradle_exec = report_dir.parent / "jacoco.exec"
        if gradle_exec.exists():
            return gradle_exec.resolve()
    return exec_files[-1] if exec_files else None


def coverage_percent(covered: int, missed: int) -> float:
    total = covered + missed
    if total == 0:
        return 100.0
    return round(covered * 100.0 / total, 2)


def parse_counter_map(xml_path: Path) -> dict[str, dict[str, int]]:
    root = ET.parse(xml_path).getroot()
    counters: dict[str, dict[str, int]] = {}
    for counter in root.findall("counter"):
        counter_type = counter.get("type", "")
        counters[counter_type] = {
            "missed": int(counter.get("missed", "0")),
            "covered": int(counter.get("covered", "0")),
        }
    return counters


def counters_to_metrics_rows(counters: dict[str, dict[str, int]]) -> list[dict[str, Any]]:
    label_map = {
        "INSTRUCTION": "Instruction",
        "BRANCH": "Branch",
        "LINE": "Line",
        "METHOD": "Method",
        "CLASS": "Class",
        "COMPLEXITY": "Complexity",
    }
    rows: list[dict[str, Any]] = []
    for counter_type in COUNTER_TYPES:
        values = counters.get(counter_type, {"missed": 0, "covered": 0})
        covered = values.get("covered", 0)
        missed = values.get("missed", 0)
        rows.append(
            {
                "metric_name": f"{label_map[counter_type]} Covered",
                "covered": covered,
                "missed": missed,
                "coverage_percent": coverage_percent(covered, missed),
            }
        )
    return rows


def element_counters(element: ET.Element) -> dict[str, dict[str, int]]:
    counters: dict[str, dict[str, int]] = {}
    for counter in element.findall("counter"):
        counter_type = counter.get("type", "")
        counters[counter_type] = {
            "missed": int(counter.get("missed", "0")),
            "covered": int(counter.get("covered", "0")),
        }
    return counters


def build_package_summary_rows(xml_path: Path) -> list[dict[str, Any]]:
    root = ET.parse(xml_path).getroot()
    rows: list[dict[str, Any]] = []
    for package in root.findall("package"):
        package_name = package.get("name", "").replace("/", ".")
        counters = element_counters(package)
        rows.append(
            {
                "package": package_name,
                "instruction_coverage": coverage_percent(
                    counters.get("INSTRUCTION", {}).get("covered", 0),
                    counters.get("INSTRUCTION", {}).get("missed", 0),
                ),
                "branch_coverage": coverage_percent(
                    counters.get("BRANCH", {}).get("covered", 0),
                    counters.get("BRANCH", {}).get("missed", 0),
                ),
                "line_coverage": coverage_percent(
                    counters.get("LINE", {}).get("covered", 0),
                    counters.get("LINE", {}).get("missed", 0),
                ),
                "method_coverage": coverage_percent(
                    counters.get("METHOD", {}).get("covered", 0),
                    counters.get("METHOD", {}).get("missed", 0),
                ),
                "class_coverage": coverage_percent(
                    counters.get("CLASS", {}).get("covered", 0),
                    counters.get("CLASS", {}).get("missed", 0),
                ),
                "complexity_coverage": coverage_percent(
                    counters.get("COMPLEXITY", {}).get("covered", 0),
                    counters.get("COMPLEXITY", {}).get("missed", 0),
                ),
            }
        )
    return rows


def build_class_summary_rows(xml_path: Path) -> list[dict[str, Any]]:
    root = ET.parse(xml_path).getroot()
    rows: list[dict[str, Any]] = []
    for package in root.findall("package"):
        package_name = package.get("name", "").replace("/", ".")
        for class_el in package.findall("class"):
            class_name = class_el.get("name", "").split("/")[-1]
            counters = element_counters(class_el)
            rows.append(
                {
                    "package": package_name,
                    "class": class_name,
                    "instruction_coverage": coverage_percent(
                        counters.get("INSTRUCTION", {}).get("covered", 0),
                        counters.get("INSTRUCTION", {}).get("missed", 0),
                    ),
                    "branch_coverage": coverage_percent(
                        counters.get("BRANCH", {}).get("covered", 0),
                        counters.get("BRANCH", {}).get("missed", 0),
                    ),
                    "line_coverage": coverage_percent(
                        counters.get("LINE", {}).get("covered", 0),
                        counters.get("LINE", {}).get("missed", 0),
                    ),
                    "method_coverage": coverage_percent(
                        counters.get("METHOD", {}).get("covered", 0),
                        counters.get("METHOD", {}).get("missed", 0),
                    ),
                    "complexity_coverage": coverage_percent(
                        counters.get("COMPLEXITY", {}).get("covered", 0),
                        counters.get("COMPLEXITY", {}).get("missed", 0),
                    ),
                }
            )
    return rows


def build_repository_metrics_row(
    xml_path: Path,
    repo_stats: dict[str, Any],
    total_execution_time: float,
) -> dict[str, Any]:
    root = ET.parse(xml_path).getroot()
    counters = parse_counter_map(xml_path)
    packages = root.findall("package")
    classes = root.findall(".//class")
    return {
        "Total Packages": len(packages),
        "Total Classes": len(classes),
        "Instruction Coverage %": coverage_percent(
            counters.get("INSTRUCTION", {}).get("covered", 0),
            counters.get("INSTRUCTION", {}).get("missed", 0),
        ),
        "Branch Coverage %": coverage_percent(
            counters.get("BRANCH", {}).get("covered", 0),
            counters.get("BRANCH", {}).get("missed", 0),
        ),
        "Line Coverage %": coverage_percent(
            counters.get("LINE", {}).get("covered", 0),
            counters.get("LINE", {}).get("missed", 0),
        ),
        "Method Coverage %": coverage_percent(
            counters.get("METHOD", {}).get("covered", 0),
            counters.get("METHOD", {}).get("missed", 0),
        ),
        "Class Coverage %": coverage_percent(
            counters.get("CLASS", {}).get("covered", 0),
            counters.get("CLASS", {}).get("missed", 0),
        ),
        "Complexity Coverage %": coverage_percent(
            counters.get("COMPLEXITY", {}).get("covered", 0),
            counters.get("COMPLEXITY", {}).get("missed", 0),
        ),
        "Execution Time (seconds)": total_execution_time,
        "Repository Name": repo_stats["repository_name"],
        "Build Tool": repo_stats["build_tool"],
    }


def copy_artifact(source: Path | None, destination: Path) -> bool:
    if source is None or not source.exists():
        return False
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    return True


def execute_build_and_jacoco(
    repo_path: Path,
    build_tool: str,
    env: dict[str, str],
    logger: NotebookLogger,
) -> tuple[BuildStatus, str]:
    chunks: list[str] = []
    status = BuildStatus(build_tool=build_tool, build_command=[], jacoco_command=[])

    if build_tool == "Maven":
        maven = resolve_maven_command(repo_path, logger)
        status.build_command = [*maven, "clean", "test"]
        status.jacoco_command = [*maven, "clean", "test", "jacoco:report"]
    else:
        gradle = resolve_gradle_command(repo_path, logger)
        status.build_command = [*gradle, "clean", "test"]
        status.jacoco_command = [*gradle, "clean", "test", "jacocoTestReport"]

    build_result = run_shell_command(status.build_command, repo_path, env, logger, "build")
    status.build_result = build_result
    status.build_success = build_result.exit_code == 0
    status.test_success = "BUILD SUCCESS" in combine_streams(build_result.stdout, build_result.stderr) or build_result.exit_code == 0
    chunks.append(f"===== {' '.join(status.build_command)} =====")
    chunks.append(combine_streams(build_result.stdout, build_result.stderr))

    report_dirs = find_jacoco_report_dirs(repo_path)
    if not report_dirs and build_tool == "Maven":
        for module_pom in repo_path.rglob("pom.xml"):
            module_dir = module_pom.parent
            if module_dir == repo_path:
                continue
            module_text = module_pom.read_text(encoding="utf-8", errors="replace")
            if "jacoco-maven-plugin" not in module_text:
                continue
            jacoco_result = run_shell_command(
                [*resolve_maven_command(repo_path, logger), "jacoco:report"],
                module_dir,
                env,
                logger,
                "jacoco",
            )
            status.jacoco_result = jacoco_result
            chunks.append(f"===== module jacoco report {module_dir} =====")
            chunks.append(combine_streams(jacoco_result.stdout, jacoco_result.stderr))
        report_dirs = find_jacoco_report_dirs(repo_path)
        status.test_success = status.test_success or (status.jacoco_result.exit_code == 0 if status.jacoco_result else False)

    exec_files = find_jacoco_exec_files(repo_path)
    status.report_dir = choose_primary_report_dir(report_dirs)
    status.jacoco_exec = choose_primary_exec(exec_files, status.report_dir)
    if status.report_dir is not None:
        status.jacoco_xml = status.report_dir / "jacoco.xml"
        status.jacoco_csv = status.report_dir / "jacoco.csv"
        status.index_html = status.report_dir / "index.html"
        status.report_generated = status.jacoco_xml.exists()
    if not status.report_generated:
        logger.error("JaCoCo report files were not found after build.", step="jacoco")

    return status, "\n".join(chunks)


def collect_outputs(
    status: BuildStatus,
    repo_stats: dict[str, Any],
    output_dir: Path,
    total_execution_time: float,
    logger: NotebookLogger,
) -> dict[str, Any]:
    ensure_output_dir(output_dir)
    copied = {
        "jacoco.exec": copy_artifact(status.jacoco_exec, output_dir / "jacoco.exec"),
        "jacoco.xml": copy_artifact(status.jacoco_xml, output_dir / "jacoco.xml"),
        "jacoco.csv": copy_artifact(status.jacoco_csv, output_dir / "jacoco.csv"),
        "index.html": copy_artifact(status.index_html, output_dir / "index.html"),
    }
    if not copied["jacoco.xml"]:
        raise FileNotFoundError("Unable to copy jacoco.xml to outputs.")

    xml_path = output_dir / "jacoco.xml"
    metrics_df = pd.DataFrame(counters_to_metrics_rows(parse_counter_map(xml_path)))
    metrics_df.to_csv(output_dir / "jacoco_metrics.csv", index=False)

    package_df = pd.DataFrame(build_package_summary_rows(xml_path))
    package_df.to_csv(output_dir / "package_summary.csv", index=False)

    class_df = pd.DataFrame(build_class_summary_rows(xml_path))
    class_df.to_csv(output_dir / "class_summary.csv", index=False)

    repository_metrics = build_repository_metrics_row(xml_path, repo_stats, total_execution_time)
    pd.DataFrame([repository_metrics]).to_csv(output_dir / "repository_metrics.csv", index=False)

    return {
        "copied": copied,
        "metrics_df": metrics_df,
        "package_df": package_df,
        "class_df": class_df,
        "repository_metrics": repository_metrics,
    }


def preview_raw_output(raw_text: str, preview_lines: int, output_path: Path) -> None:
    lines = raw_text.splitlines()
    print(f"\n{'=' * 80}")
    print(f"RAW JACOCO CONSOLE OUTPUT PREVIEW (first {preview_lines} lines)")
    print(f"{'=' * 80}\n")
    if not lines:
        print("(empty raw output)")
        return
    print("\n".join(lines[:preview_lines]))
    remaining = len(lines) - preview_lines
    if remaining > 0:
        print(f"\n... ({remaining} more lines saved to {output_path})")


"""Static DU raw output extraction helpers."""
from __future__ import annotations

import json
import re
import shutil
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import pandas as pd

TOOL_ROOT = Path(__file__).resolve().parent
JACOCO_TOOL_ROOT = TOOL_ROOT.parent.parent / "JaCoCo Coverage" / "tool"
if str(JACOCO_TOOL_ROOT) not in sys.path:
    sys.path.insert(0, str(JACOCO_TOOL_ROOT))

from _jacoco_notebook_utils import (  # noqa: E402
    CommandResult,
    NotebookLogger,
    combine_streams,
    compute_repository_stats,
    detect_build_tool,
    discover_java_files,
    ensure_output_dir,
    extract_package_name,
    resolve_gradle_command,
    resolve_maven_command,
    run_shell_command,
    save_java_inventory,
)

DATA_FLOW_METRICS = [
    "Variable Definition Detection",
    "Definition-Use Mapping",
    "Coverage Measurement",
    "Uncovered Definition Detection",
    "Variable Use Detection",
]

METRIC_EVIDENCE_KEYS: dict[str, list[str]] = {
    "Variable Definition Detection": [
        "definitions_total",
        "definitions_covered",
        "all_defs_percent",
        "all_defs_coverage_score",
    ],
    "Definition-Use Mapping": [
        "du_paths",
        "du_pairs_total",
        "du_pairs_covered",
        "data_path_correlation_percent",
        "data_path_correlation_score",
    ],
    "Coverage Measurement": [
        "du_path_percent",
        "all_uses_percent",
        "all_defs_percent",
        "du_path_validation_score",
    ],
    "Uncovered Definition Detection": [
        "uncovered_definitions",
        "dead_data_identification_score",
    ],
    "Variable Use Detection": [
        "uses_total",
        "uses_covered",
        "c_use_total",
        "p_use_total",
        "all_uses_percent",
        "all_uses_coverage_score",
    ],
}

DERIVED_SCORE_KEYS = {
    "dead_data_identification_score",
    "all_defs_coverage_score",
    "data_path_correlation_score",
    "du_path_validation_score",
    "all_uses_coverage_score",
}

STATIC_DU_MAIN_CLASS = "com.testable.training.platform.StaticDuTrigger"
STATIC_DU_PLATFORM_DIR = "static-du-platform"

USE_TYPE_MAP = {
    "computational": "C-Use",
    "return": "C-Use",
    "predicate": "P-Use",
    "c-use": "C-Use",
    "p-use": "P-Use",
}

CLASS_NAME_RE = re.compile(r"(?:public\s+)?(?:final\s+)?class\s+(\w+)")
METHOD_RE = re.compile(r"(?:public|private|protected)\s+[\w<>,\s\[\]]+\s+(\w+)\s*\(")


@dataclass
class StaticDuRunStatus:
    command: list[str]
    build_command: list[str]
    build_result: CommandResult | None
    run_result: CommandResult | None
    build_success: bool = False
    run_success: bool = False
    static_du_json: Path | None = None
    static_du_summary_json: Path | None = None
    du_path_correlation_json: Path | None = None
    static_du_meta_json: Path | None = None


def detect_static_du_platform(repo_path: Path) -> Path | None:
    platform = repo_path / STATIC_DU_PLATFORM_DIR
    if (platform / "pom.xml").exists():
        return platform.resolve()
    for pom in repo_path.rglob("pom.xml"):
        if pom.parent.name == STATIC_DU_PLATFORM_DIR:
            return pom.parent.resolve()
    return None


def resolve_static_du_command(repo_path: Path, logger: NotebookLogger, skip_verify: bool = False) -> list[str]:
    platform = detect_static_du_platform(repo_path)
    if platform is None:
        raise FileNotFoundError(
            f"No {STATIC_DU_PLATFORM_DIR} module found in repository. Cannot execute Static DU trigger."
        )
    maven = resolve_maven_command(repo_path, logger)
    command = [
        *maven,
        "-pl",
        STATIC_DU_PLATFORM_DIR,
        "exec:java",
        f"-Dexec.mainClass={STATIC_DU_MAIN_CLASS}",
    ]
    if skip_verify:
        command.append("-Dexec.args=--skip-verify")
    return command


def execute_build_only(
    repo_path: Path,
    build_tool: str,
    env: dict[str, str],
    logger: NotebookLogger,
) -> CommandResult:
    if build_tool == "Maven":
        command = [*resolve_maven_command(repo_path, logger), "clean", "compile"]
        platform = detect_static_du_platform(repo_path)
        if platform is not None:
            command = [*resolve_maven_command(repo_path, logger), "clean", "compile", "-pl", STATIC_DU_PLATFORM_DIR, "-am"]
    else:
        command = [*resolve_gradle_command(repo_path, logger), "clean", "build", "-x", "test"]
    return run_shell_command(command, repo_path, env, logger, "build")


def execute_static_du(
    repo_path: Path,
    env: dict[str, str],
    logger: NotebookLogger,
    skip_verify: bool = True,
) -> tuple[StaticDuRunStatus, str]:
    chunks: list[str] = []
    status = StaticDuRunStatus(command=[], build_command=[], build_result=None, run_result=None)
    build_tool = detect_build_tool(repo_path)

    if build_tool == "Maven":
        build_command = [*resolve_maven_command(repo_path, logger), "clean", "compile"]
        if detect_static_du_platform(repo_path) is not None:
            build_command = [
                *resolve_maven_command(repo_path, logger),
                "clean",
                "compile",
                "-pl",
                STATIC_DU_PLATFORM_DIR,
                "-am",
            ]
    else:
        build_command = [*resolve_gradle_command(repo_path, logger), "clean", "build", "-x", "test"]
    status.build_command = build_command
    build_result = run_shell_command(build_command, repo_path, env, logger, "build")
    status.build_result = build_result
    status.build_success = build_result.exit_code == 0
    chunks.append(f"===== {' '.join(build_result.command)} =====")
    chunks.append(combine_streams(build_result.stdout, build_result.stderr))

    try:
        status.command = resolve_static_du_command(repo_path, logger, skip_verify=skip_verify)
    except FileNotFoundError as exc:
        logger.error(str(exc), step="static_du_detect")
        return status, "\n".join(chunks)

    run_result = run_shell_command(status.command, repo_path, env, logger, "static_du_run")
    status.run_result = run_result
    status.run_success = run_result.exit_code == 0
    chunks.append(f"===== {' '.join(status.command)} =====")
    chunks.append(combine_streams(run_result.stdout, run_result.stderr))

    status.static_du_json = repo_path / "static_du.json"
    status.static_du_summary_json = repo_path / "artifacts" / "training" / "static_du_summary.json"
    status.du_path_correlation_json = repo_path / "artifacts" / "training" / "du_path_correlation.json"
    status.static_du_meta_json = repo_path / "artifacts" / "training" / "static_du_meta.json"
    return status, "\n".join(chunks)


def copy_artifact(source: Path | None, destination: Path) -> bool:
    if source is None or not source.exists():
        return False
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    return True


def preserve_static_du_artifacts(status: StaticDuRunStatus, output_dir: Path, repo_path: Path | None = None) -> dict[str, bool]:
    ensure_output_dir(output_dir)
    copied = {
        "static_du_output.json": copy_artifact(status.static_du_json, output_dir / "static_du_output.json"),
        "static_du_summary": copy_artifact(
            status.static_du_summary_json,
            output_dir / "static_du_summary_copy.json",
        ),
    }
    if repo_path is not None:
        for pattern in ("static_du*.xml", "static_du*.csv", "*static_du*.xml", "*static_du*.csv"):
            for source in repo_path.rglob(pattern.split("/")[-1]):
                if not source.is_file():
                    continue
                suffix = source.suffix.lower()
                if suffix == ".xml":
                    copied["static_du_output.xml"] = copy_artifact(source, output_dir / "static_du_output.xml")
                elif suffix == ".csv":
                    copied["static_du_output.csv"] = copy_artifact(source, output_dir / "static_du_output.csv")
    return copied


def load_json_map(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def merged_summary_payload(repo_path: Path, output_dir: Path) -> dict[str, Any]:
    candidates = [
        output_dir / "static_du_output.json",
        repo_path / "static_du.json",
        repo_path / "artifacts" / "training" / "static_du_summary.json",
        repo_path / "artifacts" / "training" / "du_path_correlation.json",
    ]
    merged: dict[str, Any] = {}
    for path in candidates:
        if not path.exists():
            continue
        payload = load_json_map(path)
        if "supplemental_raw_data" in payload:
            supplemental = payload.get("supplemental_raw_data", {})
            if isinstance(supplemental, dict):
                summary = supplemental.get("static_du_summary", {})
                if isinstance(summary, dict):
                    merged.update(summary)
                correlation = supplemental.get("du_path_correlation", {})
                if isinstance(correlation, dict):
                    for key, value in correlation.items():
                        merged.setdefault(key, value)
        if "summary" in payload and isinstance(payload["summary"], dict):
            merged.update(payload["summary"])
        if "definitions_total" in payload:
            merged.update(payload)
        if "du_paths" in payload and "du_paths" not in merged:
            merged["du_paths"] = payload["du_paths"]
    return merged


def map_use_type(raw_value: str) -> str:
    normalized = raw_value.strip().lower()
    return USE_TYPE_MAP.get(normalized, raw_value)


def extract_class_name(java_path: Path) -> str:
    try:
        text = java_path.read_text(encoding="utf-8")
    except (OSError, UnicodeDecodeError):
        return java_path.stem
    match = CLASS_NAME_RE.search(text)
    return match.group(1) if match else java_path.stem


def find_java_file_by_name(repo_path: Path, file_name: str) -> Path | None:
    for path in repo_path.rglob(file_name):
        if any(part in {".git", "target", "build"} for part in path.parts):
            continue
        return path.resolve()
    return None


def extract_variable_definitions(summary: dict[str, Any], repo_path: Path) -> pd.DataFrame:
    rows: list[dict[str, str]] = []
    definitions = summary.get("definitions")
    if isinstance(definitions, list):
        for item in definitions:
            if not isinstance(item, dict):
                continue
            rows.append(
                {
                    "file": str(item.get("file", "")),
                    "class": str(item.get("class", "")),
                    "method": str(item.get("method", "")),
                    "variable": str(item.get("variable", "")),
                    "definition_line": str(item.get("definition_line", item.get("line", ""))),
                    "definition_type": str(item.get("definition_type", item.get("type", ""))),
                }
            )
    return pd.DataFrame(
        rows,
        columns=["file", "class", "method", "variable", "definition_line", "definition_type"],
    )


def extract_definition_use_pairs(summary: dict[str, Any], repo_path: Path) -> pd.DataFrame:
    rows: list[dict[str, str]] = []
    du_paths = summary.get("du_paths", [])
    if not isinstance(du_paths, list):
        return pd.DataFrame(
            columns=[
                "file",
                "class",
                "method",
                "variable",
                "definition_line",
                "use_line",
                "use_type",
                "du_pair",
            ]
        )

    for item in du_paths:
        if not isinstance(item, dict):
            continue
        file_name = str(item.get("file", ""))
        variable = str(item.get("variable", ""))
        use_line = str(item.get("line", ""))
        emitted_use_type = str(item.get("use_type", ""))
        mapped_use_type = map_use_type(emitted_use_type)
        java_path = find_java_file_by_name(repo_path, file_name) if file_name else None
        class_name = extract_class_name(java_path) if java_path else ""
        du_pair = f"{variable}@{use_line}" if variable and use_line else variable
        rows.append(
            {
                "file": file_name,
                "class": class_name,
                "method": "",
                "variable": variable,
                "definition_line": "",
                "use_line": use_line,
                "use_type": mapped_use_type,
                "du_pair": du_pair,
            }
        )
    return pd.DataFrame(rows)


def collect_metric_evidence(summary: dict[str, Any], unified_json: dict[str, Any]) -> dict[str, list[str]]:
    evidence: dict[str, list[str]] = {metric: [] for metric in DATA_FLOW_METRICS}
    for key, value in summary.items():
        for metric, keys in METRIC_EVIDENCE_KEYS.items():
            if key in keys:
                evidence[metric].append(f"{key}={value}")
    metrics_rows = unified_json.get("metrics", [])
    if isinstance(metrics_rows, list):
        for row in metrics_rows:
            if not isinstance(row, dict):
                continue
            l5 = str(row.get("l5_metric", ""))
            if l5 in evidence:
                for field in ("covered", "score", "value", "formula"):
                    if field in row:
                        evidence[l5].append(f"{field}={row[field]}")
    return evidence


def validate_data_flow_metrics(summary: dict[str, Any], unified_json: dict[str, Any]) -> pd.DataFrame:
    evidence_map = collect_metric_evidence(summary, unified_json)
    rows: list[dict[str, str]] = []
    for metric in DATA_FLOW_METRICS:
        evidence_parts = evidence_map.get(metric, [])
        directly_emitted = any(
            part.split("=", 1)[0] not in DERIVED_SCORE_KEYS
            for part in evidence_parts
            if "=" in part
        )
        derived = any(
            part.split("=", 1)[0] in DERIVED_SCORE_KEYS
            for part in evidence_parts
            if "=" in part
        )
        if evidence_parts:
            supported = "Supported"
            evidence = " | ".join(evidence_parts[:6])
        else:
            supported = "Not Supported"
            evidence = ""
        rows.append(
            {
                "Metric": metric,
                "Supported": supported,
                "Directly Emitted": "Yes" if directly_emitted else "No",
                "Derived": "Yes" if derived and not directly_emitted else ("Yes" if derived else "No"),
                "Evidence": evidence,
            }
        )
    return pd.DataFrame(rows)


def build_repository_metrics(
    java_files: list[Path],
    summary: dict[str, Any],
    du_pairs_df: pd.DataFrame,
    execution_time: float,
) -> pd.DataFrame:
    classes = {extract_class_name(path) for path in java_files}
    c_uses = int((du_pairs_df["use_type"] == "C-Use").sum()) if not du_pairs_df.empty else int(summary.get("c_use_total", 0) or 0)
    p_uses_reported = int(summary.get("p_use_total", 0) or 0)
    p_uses_pairs = int((du_pairs_df["use_type"] == "P-Use").sum()) if not du_pairs_df.empty else 0
    files_emitted = summary.get("files", [])
    total_classes = len(files_emitted) if isinstance(files_emitted, list) and files_emitted else len(classes)
    return pd.DataFrame(
        [
            {
                "Total Java Files": len(java_files),
                "Total Classes": total_classes,
                "Total Methods": "",
                "Total Variable Definitions": int(summary.get("definitions_total", 0) or 0),
                "Total Variable Uses": int(summary.get("uses_total", 0) or 0),
                "Total DU Pairs": int(summary.get("du_pairs_total", 0) or 0),
                "Total C-Uses": c_uses if c_uses else int(summary.get("c_use_total", 0) or 0),
                "Total P-Uses": p_uses_reported if p_uses_reported else p_uses_pairs,
                "Execution Time": execution_time,
            }
        ]
    )


def build_dashboard_table(repo_metrics: pd.DataFrame) -> pd.DataFrame:
    if repo_metrics.empty:
        return pd.DataFrame(columns=["Metric", "Value"])
    row = repo_metrics.iloc[0].to_dict()
    return pd.DataFrame(
        [
            {"Metric": "Java Files", "Value": row.get("Total Java Files", "")},
            {"Metric": "Classes", "Value": row.get("Total Classes", "")},
            {"Metric": "Methods", "Value": row.get("Total Methods", "")},
            {"Metric": "Variable Definitions", "Value": row.get("Total Variable Definitions", "")},
            {"Metric": "Variable Uses", "Value": row.get("Total Variable Uses", "")},
            {"Metric": "Definition-Use Pairs", "Value": row.get("Total DU Pairs", "")},
            {"Metric": "C-Uses", "Value": row.get("Total C-Uses", "")},
            {"Metric": "P-Uses", "Value": row.get("Total P-Uses", "")},
        ]
    )


def preview_raw_output(raw_text: str, max_lines: int, source_path: Path) -> None:
    lines = raw_text.splitlines()
    print(f"Saved raw output: {source_path} ({len(lines)} lines)")
    preview = "\n".join(lines[:max_lines])
    print(preview)
    if len(lines) > max_lines:
        print(f"... truncated preview ({len(lines) - max_lines} additional lines in file)")


def collect_outputs(
    status: StaticDuRunStatus,
    repo_path: Path,
    java_files: list[Path],
    output_dir: Path,
    execution_time: float,
) -> dict[str, Any]:
    ensure_output_dir(output_dir)
    unified_json = load_json_map(status.static_du_json) if status.static_du_json and status.static_du_json.exists() else {}
    summary = merged_summary_payload(repo_path, output_dir)

    definitions_df = extract_variable_definitions(summary, repo_path)
    du_pairs_df = extract_definition_use_pairs(summary, repo_path)
    metrics_df = validate_data_flow_metrics(summary, unified_json)
    repository_metrics_df = build_repository_metrics(java_files, summary, du_pairs_df, execution_time)
    dashboard_df = build_dashboard_table(repository_metrics_df)

    definitions_df.to_csv(output_dir / "variable_definitions.csv", index=False)
    du_pairs_df.to_csv(output_dir / "definition_use_pairs.csv", index=False)
    metrics_df.to_csv(output_dir / "data_flow_metrics.csv", index=False)
    repository_metrics_df.to_csv(output_dir / "repository_metrics.csv", index=False)
    dashboard_df.to_csv(output_dir / "dashboard.csv", index=False)

    return {
        "definitions_df": definitions_df,
        "du_pairs_df": du_pairs_df,
        "metrics_df": metrics_df,
        "repository_metrics_df": repository_metrics_df,
        "dashboard_df": dashboard_df,
        "summary": summary,
    }



## Section 4 — Repository Information


In [ ]:
OUTPUT_PATH = Path(OUTPUT_DIR).resolve()
WORKSPACE_PATH = Path(WORKSPACE_DIR).resolve()
ERROR_LOG_PATH = OUTPUT_PATH / 'error_log.txt'

ensure_output_dir(OUTPUT_PATH)
logger = NotebookLogger(ERROR_LOG_PATH)
JAVA_ENV = configure_java_runtime(logger)
JAVA_VERSION = java_version_text(JAVA_ENV)

REPO_PATH = resolve_repository_path(
    use_git_repo=USE_GIT_REPO,
    repo_url=REPO_URL,
    local_repo=LOCAL_REPO,
    workspace_dir=WORKSPACE_PATH,
    if_clone_exists=IF_CLONE_EXISTS,
    logger=logger,
    clone_depth=CLONE_DEPTH,
)

BUILD_TOOL = detect_build_tool(REPO_PATH)
JAVA_FILES = discover_java_files(REPO_PATH)
if not JAVA_FILES:
    raise FileNotFoundError('No Java source files found.')

REPO_STATS = compute_repository_stats(REPO_PATH, JAVA_FILES, BUILD_TOOL, JAVA_VERSION)
REPOSITORY_SUMMARY_CSV = OUTPUT_PATH / 'repository_summary.csv'
pd.DataFrame([REPO_STATS]).to_csv(REPOSITORY_SUMMARY_CSV, index=False)

print(f"Repository Name: {REPO_STATS['repository_name']}")
print(f"Repository Path: {REPO_STATS['repository_location']}")
print(f"Build Tool: {REPO_STATS['build_tool']}")
print(f"Java Version: {REPO_STATS['java_version']}")
print(f"Total Java Files: {REPO_STATS['total_java_files']}")
print(f"Repository Size (bytes): {REPO_STATS['repository_size_bytes']}")


## Section 5 — Discover Java Files


In [ ]:
INVENTORY_CSV = OUTPUT_PATH / 'java_files_inventory.csv'
save_java_inventory(JAVA_FILES, INVENTORY_CSV)
display(pd.read_csv(INVENTORY_CSV).head(20))


## Section 6 — Detect Build Tool


In [ ]:
print('Detected Build Tool:')
print(BUILD_TOOL)
platform = detect_static_du_platform(REPO_PATH)
print('Static DU platform module:', platform)


## Section 7 — Build Project


In [ ]:
PIPELINE_STARTED = time.perf_counter()
STATIC_DU_STATUS, RAW_CONSOLE_OUTPUT = execute_static_du(
    REPO_PATH,
    JAVA_ENV,
    logger,
    skip_verify=SKIP_VERIFY,
)
print(f'Build success: {STATIC_DU_STATUS.build_success}')
print(f'Static DU run success: {STATIC_DU_STATUS.run_success}')


## Section 8 — Execute Static DU Tool


In [ ]:
if STATIC_DU_STATUS.command:
    print('Static DU command:')
    print(' '.join(STATIC_DU_STATUS.command))
if STATIC_DU_STATUS.run_result:
    print(f"Execution time (s): {STATIC_DU_STATUS.run_result.execution_time_seconds}")


## Section 9 — Preserve Raw Output


In [ ]:
CONSOLE_PATH = OUTPUT_PATH / 'static_du_console_output.txt'
CONSOLE_PATH.write_text(RAW_CONSOLE_OUTPUT, encoding='utf-8')
COPIED = preserve_static_du_artifacts(STATIC_DU_STATUS, OUTPUT_PATH, REPO_PATH)
print('Copied artifacts:', COPIED)
preview_raw_output(RAW_CONSOLE_OUTPUT, RAW_OUTPUT_PREVIEW_LINES, CONSOLE_PATH)


## Section 10 — Extract Variable Definitions


In [ ]:
TOTAL_EXECUTION_TIME = round(time.perf_counter() - PIPELINE_STARTED, 5)
PARSED = collect_outputs(
    STATIC_DU_STATUS,
    REPO_PATH,
    JAVA_FILES,
    OUTPUT_PATH,
    TOTAL_EXECUTION_TIME,
)
DEFINITIONS_DF = PARSED['definitions_df']
if DEFINITIONS_DF.empty:
    logger.error(
        'Static DU did not emit per-definition records; only aggregate definition counts are present in JSON.',
        step='parse_definitions',
    )
display(DEFINITIONS_DF.head(20))


## Section 11 — Extract Definition-Use Relationships


In [ ]:
DU_PAIRS_DF = PARSED['du_pairs_df']
display(DU_PAIRS_DF.head(20))


## Section 12 — Validate Required Metrics


In [ ]:
DATA_FLOW_METRICS_DF = PARSED['metrics_df']
display(DATA_FLOW_METRICS_DF)


## Section 13 — Repository Summary


In [ ]:
REPOSITORY_METRICS_DF = PARSED['repository_metrics_df']
display(REPOSITORY_METRICS_DF)


## Section 14 — Dashboard


In [ ]:
DASHBOARD_DF = PARSED['dashboard_df']
display(DASHBOARD_DF)


## Section 15 — Error Handling


In [ ]:
logger.write_errors()
if ERROR_LOG_PATH.exists():
    display(pd.read_csv(ERROR_LOG_PATH))
else:
    print('No errors logged.')
